# NB11：PLS 迴歸定量 I — 醋酸（Acetic Acid）NIR 定量校正

**食品分析實驗 週次 13｜對應 Nielsen's Food Analysis, 6th ed. 光譜法／化學計量學章節，銜接 W13 課堂 deck**

## 學習目標 (Learning Objectives)

完成本筆記本後，你應該能夠：

1. 用共用的長格式光譜資料結構（`load_spectra` / `split_spectra` / `plot_spectra`）載入並檢視 NIR 光譜。
2. 用 **Kennard–Stone 演算法** 把樣品切成校正集（calibration set）與測試集（test set），並解釋為什麼測試集絕不能拿來調參。
3. 比較不同前處理方法（不做前處理 / SNV / Savitzky–Golay 一階導數）對 **RMSECV** 的影響。
4. 畫出 **RMSEC、RMSECV vs LV 數（1–15）** 的曲線，判斷欠擬合／過擬合，並選出最佳潛在變數（latent variable, LV）數。
5. 用**分組交叉驗證**（`GroupKFold` / `LeaveOneGroupOut`，以 `sample_id` 分組）避免同一樣品的重複測量同時出現在訓練與驗證折中造成的資料外洩（data leakage）。
6. 計算最終模型的 **RMSEP、R²、bias、RPD**，並整理成一張 model report card。
7. 看懂等效的 R（`pls` 套件）與 Orange（兩個 `Test and Score`：一個算 RMSECV、一個算 RMSEP）建模流程。

## 如何使用本筆記本

- 直接「全部執行」（Run all）即可用內建的**模擬光譜資料**跑完整份筆記本。
- 若要換成你們班上 InnoSpectra／Avantes 實際量測的光譜，把資料整理成本筆記本使用的共用長格式（見下方「資料格式」），存成 CSV 後傳入 `load_spectra("你的檔案路徑或URL")`；本筆記本第 9 節示範了這個用法。
- 若你的資料是用 **SpectraView** 匯出（`layout="rows"`），`load_spectra()` 也能直接讀取並自動轉換成寬格式——但轉換後 `meta` 只會有 `sample_id`（SpectraView 匯出檔本身不含組別、儀器、`acidity_pct` 等標籤），要用 `acidity_pct` 等欄位時，需要自己另外準備一張「`sample_id` → 標籤」對照表，用 `sample_id` 做 `merge`，見第 9 節示範。

> ⚠️ **本筆記本內建的資料是模擬教學資料（模擬光譜）**，不是真實量測值，僅用於練習 PLS 建模流程與寫法。做作業時請換成你們班的實際數據。

本筆記本使用的資料欄位重點：`sample_id`（樣品唯一代號，**同一樣品的所有重複測量必須共用同一個 `sample_id`**，交叉驗證要用它分組）、`group`（本筆記本內建資料固定為 `0`，代表教師提供的示範資料）、`instrument`、`rep`、以及目標值欄位 `acidity_pct`（醋酸濃度 % w/v，本週的參考值來自 W4 手動滴定）。完整的共用光譜資料格式規範見下方 Setup 小節。

## 1. 環境設定 (Setup)

匯入套件，並定義本課程所有光譜筆記本共用的三個函式：`load_spectra()`、`split_spectra()`、`plot_spectra()`。

**注意**：圖表座標軸與圖例一律使用英文，避免 Colab 預設字型無法顯示中文而出現方框亂碼；文字說明使用繁體中文。

In [ ]:
import io
import os

import numpy as np
import pandas as pd
from scipy import stats
from scipy.signal import savgol_filter
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut, KFold

plt.style.use("ggplot")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.precision", 4)

RNG_SEED = 13
np.random.seed(RNG_SEED)

PREP_SD = 0.05  # %w/v, W4 titration reference-value noise baked into the demo data

### 光譜資料格式 (Shared Spectral Data Format) — NB09–NB13 共用

從本週開始，資料改用**寬格式 (wide format)**，跟 NB01 的長格式不同：**一列＝一條完整光譜**。

| 規則 | 說明 |
|---|---|
| 中繼資料欄位在前 | 欄名**不是數字**的欄位，例如 `sample_id`（樣品代號）、`group`（組別 1–5，沿用 NB01 的定義；TA 提供的參考/示範資料用 `0`）、`instrument`（儀器代號）、`rep`（重複次數）、`role`（`library`/`unknown`/`mixture` 等角色）、以及依主題而定的標籤欄位，例如 `class`（化合物名稱）、`oil`（油品代號）、`acidity_pct`、`aspartame_pct`、`sucralose_pct` 等。 |
| 光譜欄位在後 | 欄名**能轉成浮點數**的欄位＝光譜資料本身，欄名即該點的波長 (nm，NIR) 或波數 (cm⁻¹，Raman)。程式一律依欄名數值排序，不假設檔案裡欄位順序已經由小到大。 |
| 光譜欄位不可缺值 | 每一列在所有光譜欄位都必須有數值，不可有 NaN。 |
| **一個檔案只用一種共同 x 軸** | Raman 位移與 NIR 波長不可混在同一個檔案；不同解析度/範圍的儀器資料也請分開存檔，或先內插到同一組 x 軸再合併。 |
| 標籤欄位的值用英文代碼 | 例如油品用 `EVOO`/`COC`，化合物用 `aspartame`/`sucralose`。畫圖的圖例文字直接取自欄位值，中文會因 Colab 字型問題變成方框。 |

**也相容 SpectraView 的「rows」匯出格式**（`specview/formats/__init__.py` 的 `save_combined_csv(..., layout="rows")`）：
第一列表頭的第一格是 `x:<軸標籤>`（例如 `x:Raman shift (cm-1)`），之後每一格才是 x 值本身；
從第二列開始，每一列的第一格是**光譜名稱**、其餘是強度值。這種格式沒有額外中繼資料欄位，
只有一個 `sample_id`——如果要加上組別/儀器等資訊，請自行用 `sample_id` 做 `merge`。

共用讀取/拆解/畫圖函式（簽名固定，NB09–NB13 都一樣）：

```python
load_spectra(source=None)          # -> 寬格式 DataFrame；None 時退回內建模擬光譜
split_spectra(df)                  # -> (meta, X, axis)
plot_spectra(axis, X, labels)      # -> matplotlib Axes，依 labels 分色、圖例自動去重
```

In [ ]:
def _is_number(text):
    try:
        float(text)
        return True
    except (TypeError, ValueError):
        return False


def load_spectra(source=None):
    """讀取寬格式光譜 CSV（見上方「光譜資料格式」說明）。

    Parameters
    ----------
    source : str, path, file-like, or None
        - None：退回本筆記本內嵌的模擬光譜字串 SAMPLE_CSV（模擬光譜，非真實量測）。
        - 本機/雲端硬碟路徑，或 http(s) URL（例如已發布的 Google 試算表 CSV 連結）：
          直接用 pandas.read_csv 讀取。

    Returns
    -------
    pandas.DataFrame
        寬格式：中繼資料欄位（欄名非數字）+ 光譜欄位（欄名為數字，代表 nm 或 cm-1）。
    """
    if source is None:
        return pd.read_csv(io.StringIO(SAMPLE_CSV))
    return pd.read_csv(source)


def split_spectra(df):
    """把 load_spectra() 讀進來的寬格式 DataFrame 拆成 (meta, X, axis)。

    自動判斷兩種輸入：
    - 一般格式：欄名可轉成浮點數的欄位＝光譜欄位；其餘欄位＝中繼資料 (meta)。
    - SpectraView「rows」匯出格式：第一個欄名以 "x:" 開頭，代表該欄其實是
      「樣品名稱」，其餘欄名本身就是 x 值；此時 meta 只有一欄 sample_id
      （由第一欄改名而來），其他中繼資料需自行用 sample_id 合併(merge)。

    Returns
    -------
    meta : pandas.DataFrame   中繼資料（含 sample_id 或其他非光譜欄位）
    X    : numpy.ndarray, shape (n_spectra, n_points)   光譜強度矩陣
    axis : numpy.ndarray, shape (n_points,)             x 軸，已由小到大排序
    """
    cols = list(df.columns)

    if isinstance(cols[0], str) and cols[0].startswith("x:"):
        meta = df.iloc[:, [0]].rename(columns={cols[0]: "sample_id"}).reset_index(drop=True)
        spec_cols = cols[1:]
    else:
        spec_cols = [c for c in cols if _is_number(c)]
        meta_cols = [c for c in cols if c not in spec_cols]
        meta = df[meta_cols].reset_index(drop=True)

    axis = np.array([float(c) for c in spec_cols])
    X = df[spec_cols].to_numpy(dtype=float)

    order = np.argsort(axis)
    axis = axis[order]
    X = X[:, order]
    return meta, X, axis


def plot_spectra(axis, X, labels, ax=None, title=None, xlabel="x", ylabel="Intensity (a.u.)"):
    """畫出多條光譜，依 labels 分色，圖例只顯示每個 label 一次。"""
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 5))
    labels = list(labels)
    unique_labels = sorted(set(labels), key=labels.index)
    cmap = plt.get_cmap("tab10")
    color_of = {lab: cmap(i % 10) for i, lab in enumerate(unique_labels)}
    seen = set()
    for i in range(X.shape[0]):
        lab = labels[i]
        show_label = lab if lab not in seen else None
        seen.add(lab)
        ax.plot(axis, X[i], color=color_of[lab], linewidth=1.3, alpha=0.85, label=show_label)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    ax.legend(loc="best", fontsize=8)
    return ax

In [ ]:
# ---- Embedded demo spectra (fallback) ---------------------------------
# Exact content of data/nb11_demo_spectra.csv, embedded so the notebook
# is self-contained even without the data/ folder present.
SAMPLE_CSV = """sample_id,group,instrument,rep,acidity_pct,900,913.56,927.12,940.68,954.24,967.8,981.36,994.92,1008.47,1022.03,1035.59,1049.15,1062.71,1076.27,1089.83,1103.39,1116.95,1130.51,1144.07,1157.63,1171.19,1184.75,1198.31,1211.86,1225.42,1238.98,1252.54,1266.1,1279.66,1293.22,1306.78,1320.34,1333.9,1347.46,1361.02,1374.58,1388.14,1401.69,1415.25,1428.81,1442.37,1455.93,1469.49,1483.05,1496.61,1510.17,1523.73,1537.29,1550.85,1564.41,1577.97,1591.53,1605.08,1618.64,1632.2,1645.76,1659.32,1672.88,1686.44,1700
STD-00,0,InnoSpectra_NIR-S_demo,1,0.0,0.334313,0.307519,0.301934,0.324308,0.325515,0.315833,0.321425,0.341589,0.345059,0.328419,0.334295,0.31356,0.376553,0.347926,0.378213,0.366341,0.396119,0.393168,0.37665,0.400934,0.364428,0.406895,0.365024,0.400225,0.41624,0.405714,0.400384,0.399858,0.418546,0.459204,0.482528,0.501646,0.540081,0.647402,0.706553,0.835574,0.956884,1.107439,1.212089,1.303586,1.350985,1.341471,1.324174,1.262626,1.134201,1.028226,0.897782,0.81046,0.714736,0.648801,0.58979,0.578384,0.524006,0.513405,0.504167,0.523615,0.503897,0.506007,0.532461,0.543298
STD-00,0,InnoSpectra_NIR-S_demo,2,0.0,0.331118,0.330105,0.328852,0.343473,0.382328,0.347229,0.340978,0.366385,0.356706,0.358606,0.345906,0.35858,0.367018,0.328166,0.33432,0.381192,0.378208,0.356524,0.349292,0.365298,0.350274,0.366998,0.402837,0.378509,0.397133,0.374592,0.386665,0.408166,0.370013,0.401073,0.427657,0.471131,0.545206,0.602424,0.657861,0.789409,0.900896,1.023189,1.183804,1.216264,1.286267,1.274027,1.260193,1.172408,1.062067,0.938531,0.838929,0.694097,0.605302,0.526481,0.515401,0.451655,0.415949,0.412696,0.454753,0.406727,0.421549,0.414788,0.407094,0.396273
STD-00,0,InnoSpectra_NIR-S_demo,3,0.0,0.205612,0.290279,0.265047,0.254379,0.252023,0.245134,0.277779,0.261536,0.275936,0.247278,0.280377,0.311026,0.275175,0.296779,0.286868,0.308168,0.310702,0.296916,0.289905,0.308163,0.308164,0.301216,0.323521,0.326657,0.307289,0.32363,0.379246,0.349747,0.347692,0.405027,0.379993,0.383695,0.447892,0.528507,0.653364,0.778287,0.848921,1.029583,1.10234,1.214576,1.26807,1.273522,1.227724,1.153394,1.026669,0.932752,0.826491,0.717983,0.656433,0.560285,0.503522,0.485437,0.440246,0.402507,0.380849,0.438975,0.427527,0.434108,0.416523,0.430841
STD-01,0,InnoSpectra_NIR-S_demo,1,0.7692,0.300929,0.308233,0.326876,0.365619,0.355896,0.343836,0.337115,0.359182,0.360434,0.355423,0.362502,0.355132,0.399965,0.330075,0.373294,0.355017,0.385939,0.373043,0.388903,0.400766,0.407499,0.434436,0.405452,0.451566,0.431707,0.437632,0.452887,0.444262,0.463954,0.405495,0.45165,0.522046,0.541729,0.634686,0.766802,0.871458,0.966878,1.111872,1.219379,1.288189,1.329742,1.33705,1.269355,1.219428,1.096257,1.0014,0.883945,0.754012,0.689486,0.615009,0.557494,0.536388,0.497489,0.485951,0.486211,0.481784,0.502309,0.443832,0.480818,0.454044
STD-01,0,InnoSpectra_NIR-S_demo,2,0.7692,0.218315,0.238243,0.23072,0.182197,0.21324,0.262148,0.265604,0.244046,0.219117,0.285395,0.215601,0.259057,0.25527,0.262367,0.240318,0.255592,0.283489,0.2611,0.25269,0.286102,0.3013,0.312188,0.313707,0.343804,0.333651,0.312669,0.332857,0.328501,0.337737,0.320944,0.362156,0.422,0.486595,0.554689,0.652711,0.78243,0.914702,1.050203,1.133644,1.195865,1.259307,1.222396,1.218029,1.133149,1.03697,0.928955,0.804618,0.67281,0.602481,0.526015,0.489756,0.463271,0.420684,0.438723,0.44368,0.416021,0.434293,0.399169,0.418255,0.447212
STD-01,0,InnoSpectra_NIR-S_demo,3,0.7692,0.162932,0.200639,0.203504,0.176315,0.191893,0.148194,0.209838,0.204001,0.223142,0.162354,0.211038,0.201066,0.164374,0.198511,0.210333,0.226909,0.218762,0.210268,0.232763,0.232911,0.236483,0.244282,0.27619,0.283115,0.312309,0.26957,0.300304,0.278229,0.297889,0.276499,0.301694,0.349584,0.425871,0.490989,0.601294,0.713231,0.852481,0.959796,1.052257,1.14909,1.186798,1.218691,1.146983,1.074474,0.963338,0.870725,0.748388,0.637638,0.515766,0.441577,0.42162,0.395741,0.387878,0.320689,0.342937,0.315506,0.328522,0.342901,0.365448,0.342835
STD-02,0,InnoSpectra_NIR-S_demo,1,1.5385,0.175782,0.206119,0.196816,0.218777,0.178588,0.199308,0.194748,0.187526,0.207582,0.194932,0.257831,0.235227,0.219853,0.23783,0.290622,0.267938,0.304408,0.320693,0.349387,0.363556,0.371728,0.384669,0.37544,0.40276,0.390576,0.358197,0.337101,0.300344,0.349965,0.309289,0.318951,0.361282,0.423227,0.517544,0.568425,0.743073,0.853621,0.980001,1.099011,1.167867,1.184563,1.19686,1.190974,1.105728,0.994029,0.862776,0.762275,0.646491,0.541492,0.464794,0.440946,0.405305,0.398928,0.358958,0.385887,0.354478,0.382876,0.372834,0.356365,0.352991
STD-02,0,InnoSpectra_NIR-S_demo,2,1.5385,0.26723,0.242653,0.232451,0.260447,0.296674,0.295131,0.260137,0.26973,0.283928,0.272042,0.284989,0.291722,0.287565,0.305912,0.321227,0.348376,0.340821,0.380897,0.382461,0.407371,0.424558,0.531308,0.495387,0.494842,0.470456,0.428748,0.409131,0.394348,0.389123,0.396391,0.430991,0.389604,0.485158,0.584964,0.672127,0.800177,0.957987,1.054864,1.180588,1.222668,1.268716,1.2489,1.193935,1.144057,1.048904,0.925341,0.791584,0.719512,0.620644,0.550586,0.487445,0.495469,0.472169,0.441063,0.428738,0.433664,0.459052,0.480023,0.435089,0.43056
STD-02,0,InnoSpectra_NIR-S_demo,3,1.5385,0.071502,0.110989,0.110522,0.130588,0.099181,0.092241,0.128468,0.139307,0.158674,0.117967,0.147451,0.160639,0.153804,0.187765,0.219887,0.223311,0.21866,0.269908,0.301369,0.320554,0.350105,0.330685,0.38026,0.383279,0.354028,0.338302,0.304034,0.322333,0.291019,0.287493,0.328186,0.364499,0.431344,0.487323,0.597472,0.742836,0.906851,0.998416,1.068655,1.170547,1.198387,1.185185,1.168432,1.096436,1.012725,0.914144,0.797034,0.687963,0.580616,0.524329,0.473943,0.468057,0.415279,0.408498,0.440906,0.426621,0.412691,0.42788,0.43296,0.439263
STD-03,0,InnoSpectra_NIR-S_demo,1,2.3077,0.193664,0.179627,0.166007,0.174172,0.16483,0.185507,0.216048,0.214263,0.205947,0.201392,0.187329,0.173716,0.199207,0.195975,0.222838,0.217192,0.212109,0.240423,0.299151,0.2586,0.325182,0.349598,0.364124,0.363382,0.368676,0.350784,0.335442,0.3156,0.301837,0.291041,0.328328,0.35185,0.398931,0.514862,0.599713,0.722637,0.877685,0.958865,1.080887,1.154407,1.175642,1.186981,1.179798,1.091064,0.97992,0.858284,0.723223,0.624072,0.533136,0.469794,0.421774,0.37286,0.393684,0.337241,0.35591,0.365856,0.348471,0.339453,0.353871,0.389115
STD-03,0,InnoSpectra_NIR-S_demo,2,2.3077,0.293181,0.281607,0.254529,0.2919,0.306711,0.309887,0.290566,0.26939,0.315956,0.295605,0.314009,0.324474,0.315285,0.313186,0.308709,0.317943,0.333371,0.335892,0.366766,0.352037,0.399973,0.450375,0.465514,0.478785,0.460715,0.470324,0.462051,0.398556,0.428128,0.390209,0.437958,0.438245,0.537599,0.612719,0.726284,0.876687,0.999437,1.104491,1.181409,1.261099,1.284591,1.289446,1.282091,1.195671,1.083015,0.972952,0.821798,0.742687,0.623787,0.572674,0.540853,0.507937,0.493158,0.472693,0.465782,0.4635,0.488244,0.488446,0.444173,0.474005
STD-03,0,InnoSpectra_NIR-S_demo,3,2.3077,0.315921,0.303598,0.294826,0.318736,0.310649,0.316016,0.295483,0.301999,0.321488,0.318733,0.338741,0.315173,0.349286,0.351536,0.348522,0.363745,0.358834,0.367779,0.387348,0.390957,0.443649,0.49151,0.477065,0.526563,0.496231,0.479748,0.462698,0.446749,0.458569,0.439324,0.491489,0.499854,0.564445,0.650706,0.739509,0.863976,1.018274,1.143899,1.237154,1.281413,1.353612,1.322003,1.273059,1.228257,1.126861,0.974277,0.888718,0.747892,0.681928,0.651346,0.58549,0.542694,0.521584,0.485547,0.500213,0.500908,0.476631,0.499708,0.508212,0.505517
STD-04,0,InnoSpectra_NIR-S_demo,1,3.0769,0.196368,0.210963,0.190384,0.145378,0.211803,0.201136,0.191519,0.210271,0.209635,0.182925,0.183953,0.226428,0.172679,0.224699,0.22305,0.238316,0.257428,0.261326,0.283868,0.301018,0.335106,0.386877,0.401374,0.398283,0.4004,0.34497,0.328415,0.345726,0.306489,0.297591,0.309153,0.325192,0.403496,0.48487,0.614572,0.753824,0.887135,0.967823,1.107383,1.147308,1.171088,1.163424,1.136005,1.073473,0.960236,0.814729,0.697567,0.573581,0.527646,0.469127,0.428177,0.37656,0.367644,0.32678,0.345737,0.333082,0.324351,0.308767,0.359161,0.303821
STD-04,0,InnoSpectra_NIR-S_demo,2,3.0769,0.311171,0.29412,0.318991,0.305178,0.293792,0.305435,0.289506,0.32424,0.339947,0.332059,0.329426,0.352217,0.313192,0.345227,0.33899,0.334693,0.358051,0.339907,0.375776,0.413255,0.427006,0.48486,0.530502,0.549815,0.505473,0.507051,0.452286,0.431712,0.407704,0.433086,0.426505,0.479119,0.538042,0.629784,0.715743,0.884371,1.007524,1.143519,1.195401,1.248587,1.288319,1.281872,1.243817,1.148842,1.03253,0.927369,0.82663,0.73873,0.637937,0.558364,0.522489,0.492943,0.518263,0.481551,0.457515,0.427836,0.44997,0.44935,0.466138,0.495238
STD-04,0,InnoSpectra_NIR-S_demo,3,3.0769,0.295785,0.270905,0.31532,0.321507,0.310078,0.280783,0.304664,0.329977,0.293944,0.302589,0.318652,0.324414,0.318986,0.325159,0.324924,0.314349,0.327806,0.343645,0.337924,0.388213,0.424816,0.472006,0.501751,0.489042,0.472304,0.467183,0.424869,0.400933,0.375506,0.365115,0.40902,0.437013,0.505944,0.598366,0.713811,0.848808,0.950404,1.041667,1.148454,1.194283,1.222426,1.252829,1.172788,1.138337,1.03792,0.947918,0.803652,0.71537,0.579605,0.509328,0.486962,0.413718,0.433838,0.397254,0.361988,0.424028,0.37292,0.434926,0.368538,0.38066
STD-05,0,InnoSpectra_NIR-S_demo,1,3.8462,0.219549,0.240361,0.252625,0.271901,0.26482,0.259898,0.262365,0.298168,0.263241,0.286003,0.247857,0.317412,0.300118,0.266314,0.310732,0.329286,0.327525,0.328589,0.365362,0.375415,0.432981,0.48385,0.532172,0.528336,0.554409,0.491732,0.432167,0.412026,0.403841,0.398739,0.434722,0.446441,0.498781,0.610094,0.746879,0.85448,0.984279,1.110633,1.196794,1.231768,1.280088,1.260342,1.251073,1.141486,1.045883,0.949226,0.845818,0.746451,0.632233,0.576779,0.53696,0.519558,0.460365,0.443618,0.468387,0.455075,0.483648,0.463774,0.472103,0.456783
STD-05,0,InnoSpectra_NIR-S_demo,2,3.8462,0.226737,0.215131,0.254882,0.239035,0.24969,0.241049,0.273648,0.249417,0.25833,0.271103,0.272002,0.24176,0.292348,0.269095,0.298186,0.293095,0.306215,0.310162,0.328946,0.387164,0.438308,0.505083,0.538704,0.533497,0.550844,0.532005,0.464019,0.414483,0.383849,0.402791,0.395592,0.470646,0.510715,0.597772,0.736145,0.857102,0.989633,1.122761,1.228049,1.244797,1.302754,1.255078,1.246787,1.182985,1.10509,0.932103,0.847922,0.730741,0.665163,0.583785,0.546916,0.500541,0.484834,0.490996,0.505578,0.483649,0.484233,0.477579,0.51614,0.460626
STD-05,0,InnoSpectra_NIR-S_demo,3,3.8462,0.180978,0.192242,0.188488,0.194283,0.186977,0.178913,0.171564,0.199821,0.178825,0.204691,0.21282,0.19246,0.19804,0.199096,0.22312,0.218315,0.211168,0.239586,0.283152,0.298343,0.343849,0.386465,0.417336,0.447231,0.432108,0.387501,0.368277,0.308444,0.290493,0.295688,0.318361,0.314466,0.407463,0.512787,0.59525,0.765312,0.889394,0.990473,1.087075,1.130986,1.124554,1.146324,1.129787,1.030797,0.926804,0.818643,0.702618,0.618003,0.503193,0.429265,0.406464,0.382363,0.356527,0.34706,0.316254,0.300458,0.313732,0.326701,0.300628,0.326579
STD-06,0,InnoSpectra_NIR-S_demo,1,4.6154,0.208648,0.234854,0.233164,0.218275,0.232956,0.245936,0.243675,0.210243,0.221357,0.226631,0.215328,0.261244,0.238392,0.240721,0.240584,0.263899,0.275739,0.291885,0.327741,0.333771,0.434044,0.50979,0.506591,0.525952,0.522374,0.486815,0.40775,0.35522,0.38003,0.33407,0.3136,0.406015,0.475248,0.581244,0.712707,0.888042,0.969951,1.069134,1.128416,1.223073,1.20697,1.177903,1.127808,1.086449,0.972941,0.868657,0.732065,0.627923,0.559287,0.478591,0.441524,0.395654,0.371814,0.37073,0.353095,0.359447,0.364022,0.348998,0.357497,0.363366
STD-06,0,InnoSpectra_NIR-S_demo,2,4.6154,0.279802,0.287074,0.277542,0.292432,0.277522,0.262699,0.30409,0.267601,0.301706,0.333849,0.323717,0.297857,0.33115,0.326456,0.312417,0.330251,0.331488,0.350765,0.389259,0.40114,0.500115,0.5824,0.594376,0.598318,0.589286,0.571841,0.490142,0.464437,0.472378,0.420312,0.452768,0.47977,0.545825,0.649276,0.76625,0.940766,1.080791,1.205118,1.265451,1.302699,1.343008,1.307545,1.27908,1.180416,1.112644,0.960561,0.873805,0.740279,0.668349,0.578641,0.534652,0.49893,0.502451,0.493154,0.492322,0.485375,0.476432,0.493676,0.511793,0.482584
STD-06,0,InnoSpectra_NIR-S_demo,3,4.6154,0.237259,0.205372,0.205264,0.197741,0.206536,0.206836,0.210398,0.248511,0.256109,0.240361,0.244414,0.260529,0.251066,0.25673,0.269498,0.264484,0.29589,0.344831,0.318063,0.366922,0.407013,0.476395,0.534469,0.558167,0.529653,0.467986,0.44257,0.385142,0.378762,0.367245,0.387668,0.427239,0.45572,0.608014,0.70367,0.879714,1.028055,1.117109,1.172508,1.247934,1.243777,1.252068,1.219059,1.126443,1.002966,0.905145,0.809814,0.682999,0.624019,0.515442,0.47634,0.430547,0.424401,0.455555,0.435081,0.420198,0.415646,0.448324,0.430194,0.419142
STD-07,0,InnoSpectra_NIR-S_demo,1,5.3846,0.261541,0.240779,0.277323,0.268537,0.248799,0.258016,0.270405,0.256602,0.310338,0.322443,0.32996,0.314277,0.331367,0.354345,0.341647,0.335221,0.33211,0.360245,0.414139,0.444286,0.551834,0.628225,0.657024,0.638544,0.639992,0.612687,0.519089,0.465648,0.44861,0.488286,0.450795,0.462681,0.563102,0.662472,0.805362,0.954966,1.107896,1.199276,1.285553,1.305525,1.334125,1.307511,1.304779,1.212672,1.103223,1.000478,0.870148,0.777967,0.71825,0.649847,0.607741,0.575494,0.541093,0.538809,0.566315,0.533169,0.511838,0.541686,0.539461,0.549636
STD-07,0,InnoSpectra_NIR-S_demo,2,5.3846,0.284462,0.276974,0.292416,0.273401,0.271754,0.28534,0.325905,0.2735,0.329934,0.317679,0.344142,0.321833,0.336361,0.335155,0.363409,0.324976,0.335397,0.331989,0.378913,0.409836,0.486943,0.566839,0.624874,0.618531,0.594419,0.573306,0.479362,0.437442,0.397786,0.367782,0.406021,0.439964,0.531877,0.564439,0.722053,0.899738,1.025865,1.100907,1.207272,1.233633,1.233126,1.222614,1.186278,1.125776,1.018797,0.897152,0.80052,0.709048,0.594515,0.563742,0.489065,0.465895,0.442507,0.432372,0.422347,0.378247,0.400593,0.432844,0.402276,0.424025
STD-07,0,InnoSpectra_NIR-S_demo,3,5.3846,0.21417,0.195171,0.179337,0.186371,0.172137,0.212903,0.206489,0.22955,0.238066,0.227147,0.222371,0.222274,0.237426,0.243963,0.26561,0.226278,0.25459,0.243213,0.280245,0.352012,0.414383,0.468431,0.521786,0.570197,0.524362,0.46655,0.388988,0.395538,0.352886,0.322885,0.365395,0.370739,0.416938,0.527081,0.676635,0.792471,0.978341,1.080386,1.14544,1.162693,1.190423,1.134738,1.121783,1.065218,0.956809,0.838919,0.700077,0.615283,0.502557,0.460662,0.42197,0.375143,0.372049,0.352891,0.362674,0.340061,0.319392,0.326893,0.337615,0.3531
STD-08,0,InnoSpectra_NIR-S_demo,1,6.1538,0.334373,0.321707,0.312522,0.303275,0.333167,0.315997,0.339396,0.361755,0.33595,0.339427,0.346856,0.330983,0.375226,0.34132,0.359734,0.358438,0.351829,0.402924,0.420305,0.4584,0.533385,0.592102,0.668165,0.72079,0.692864,0.599134,0.551159,0.495299,0.454546,0.451146,0.409337,0.478259,0.518758,0.643093,0.757642,0.934737,1.094569,1.150604,1.216055,1.239427,1.259926,1.221011,1.216015,1.157946,1.018945,0.919021,0.752829,0.676009,0.623371,0.552599,0.51308,0.494309,0.455396,0.440107,0.432201,0.440936,0.404129,0.466685,0.406882,0.418889
STD-08,0,InnoSpectra_NIR-S_demo,2,6.1538,0.239662,0.259931,0.231325,0.232006,0.256499,0.232308,0.259088,0.268457,0.269009,0.233684,0.268441,0.241966,0.295383,0.27782,0.307032,0.319093,0.284744,0.343125,0.370386,0.446356,0.476439,0.558445,0.630253,0.634676,0.641911,0.565773,0.509016,0.451617,0.424664,0.378497,0.426851,0.462104,0.494988,0.600141,0.726486,0.936744,1.087512,1.149911,1.214425,1.249374,1.265772,1.241861,1.228368,1.109252,1.040152,0.915926,0.808936,0.692234,0.593421,0.551526,0.524611,0.47453,0.444109,0.462463,0.462914,0.447506,0.465419,0.478136,0.478817,0.495076
STD-08,0,InnoSpectra_NIR-S_demo,3,6.1538,0.292603,0.280088,0.32988,0.322096,0.336644,0.309022,0.32836,0.315824,0.317681,0.311573,0.335684,0.350943,0.359389,0.353784,0.349012,0.393534,0.39554,0.40578,0.426169,0.501599,0.543154,0.615748,0.71056,0.724261,0.712063,0.618372,0.563428,0.480096,0.455612,0.458962,0.462533,0.482201,0.534713,0.650664,0.781244,0.936425,1.11019,1.191935,1.279972,1.318967,1.313267,1.329932,1.218388,1.13381,1.060027,0.920424,0.841469,0.753786,0.669177,0.609162,0.544523,0.518037,0.50655,0.493065,0.506852,0.484491,0.492636,0.484468,0.515736,0.504887
STD-09,0,InnoSpectra_NIR-S_demo,1,6.9231,0.125989,0.171748,0.142639,0.155617,0.151697,0.144011,0.140538,0.127315,0.162186,0.142057,0.198761,0.161158,0.176132,0.189405,0.191365,0.219645,0.209492,0.223471,0.26876,0.359845,0.432019,0.476754,0.576608,0.588127,0.572921,0.512189,0.418627,0.359384,0.335452,0.263715,0.303514,0.339329,0.408238,0.503653,0.663646,0.819175,0.979068,1.042376,1.117047,1.112969,1.150693,1.158487,1.084704,1.017372,0.922133,0.819862,0.697031,0.602291,0.487168,0.429123,0.358965,0.390545,0.31544,0.342222,0.326757,0.326937,0.307972,0.330949,0.337202,0.346871
STD-09,0,InnoSpectra_NIR-S_demo,2,6.9231,0.211819,0.252097,0.224292,0.299873,0.237277,0.23529,0.252996,0.251296,0.275609,0.281501,0.254006,0.256909,0.259267,0.283794,0.296833,0.297758,0.295214,0.319683,0.369649,0.442912,0.506538,0.618385,0.663187,0.696146,0.681247,0.576838,0.4795,0.420836,0.395196,0.366718,0.393323,0.452999,0.486394,0.616448,0.755017,0.894175,1.080915,1.176397,1.22378,1.230749,1.239586,1.213676,1.16552,1.104117,1.016814,0.881985,0.807142,0.67014,0.598127,0.504516,0.507031,0.462963,0.455211,0.404584,0.446998,0.431905,0.416724,0.424543,0.410411,0.437264
STD-09,0,InnoSpectra_NIR-S_demo,3,6.9231,0.325787,0.301711,0.279731,0.324391,0.355179,0.353196,0.324204,0.359618,0.358559,0.349964,0.340273,0.365646,0.350101,0.347867,0.375824,0.351626,0.384987,0.41912,0.4208,0.497784,0.58824,0.639775,0.730582,0.752602,0.721249,0.643945,0.596813,0.489707,0.460424,0.457944,0.484761,0.464669,0.531313,0.636268,0.794442,0.986314,1.119782,1.220953,1.288991,1.276682,1.291951,1.309821,1.245977,1.146165,1.044421,0.960797,0.809082,0.710517,0.653245,0.596002,0.554356,0.478298,0.485389,0.410935,0.449705,0.51433,0.493385,0.482287,0.458061,0.456108
STD-10,0,InnoSpectra_NIR-S_demo,1,7.6923,0.296907,0.345778,0.295955,0.335258,0.312464,0.317132,0.321611,0.344611,0.352313,0.369229,0.345723,0.364972,0.346886,0.342193,0.364664,0.363738,0.338844,0.388324,0.39646,0.478527,0.618072,0.71374,0.765457,0.804423,0.780147,0.707419,0.613162,0.491995,0.455112,0.43988,0.405591,0.4671,0.549311,0.643122,0.788394,0.956974,1.122562,1.225615,1.258044,1.294966,1.292755,1.243791,1.239436,1.150157,1.064204,0.929133,0.844577,0.71624,0.661122,0.577597,0.526311,0.476571,0.452066,0.483309,0.444573,0.44219,0.469511,0.464877,0.476744,0.447201
STD-10,0,InnoSpectra_NIR-S_demo,2,7.6923,0.189527,0.175531,0.187739,0.194279,0.194432,0.201503,0.202565,0.216845,0.2244,0.21161,0.234267,0.223524,0.220313,0.217029,0.233663,0.21824,0.249276,0.232908,0.301311,0.334337,0.455114,0.546314,0.624251,0.639721,0.621062,0.549977,0.415363,0.365123,0.321152,0.310256,0.314535,0.323172,0.420778,0.493273,0.668,0.840919,0.98308,1.071006,1.11988,1.159764,1.149354,1.115298,1.094996,1.036747,0.946456,0.769732,0.699855,0.600444,0.487909,0.404072,0.409391,0.348207,0.308871,0.3071,0.332342,0.35526,0.334441,0.305631,0.343798,0.292946
STD-10,0,InnoSpectra_NIR-S_demo,3,7.6923,0.304897,0.296261,0.304822,0.294231,0.294418,0.309317,0.350494,0.316933,0.333038,0.331523,0.353856,0.35166,0.377869,0.379842,0.362818,0.362366,0.354935,0.354375,0.425746,0.494216,0.569634,0.69479,0.786412,0.781658,0.777626,0.690486,0.58158,0.515182,0.478474,0.446081,0.458284,0.516324,0.567527,0.672809,0.80838,1.008603,1.17902,1.232514,1.2592,1.287206,1.30331,1.276566,1.224052,1.208476,1.076185,0.95985,0.815088,0.78005,0.65483,0.63948,0.58064,0.544322,0.518797,0.529398,0.49965,0.515616,0.502761,0.509035,0.544999,0.54117
STD-11,0,InnoSpectra_NIR-S_demo,1,8.4615,0.266512,0.301444,0.286348,0.304706,0.280232,0.324204,0.305651,0.307955,0.312607,0.281715,0.323625,0.305581,0.298334,0.327801,0.355148,0.320182,0.371441,0.331955,0.422102,0.482708,0.599887,0.74225,0.801313,0.791861,0.80413,0.673306,0.579448,0.487627,0.4516,0.426848,0.464587,0.475443,0.545195,0.637762,0.816955,0.96388,1.11545,1.247018,1.275799,1.269736,1.307231,1.238416,1.236267,1.164743,1.01944,0.94036,0.834257,0.718296,0.634618,0.570807,0.509973,0.469845,0.487947,0.467851,0.459793,0.467147,0.444997,0.449778,0.477402,0.466324
STD-11,0,InnoSpectra_NIR-S_demo,2,8.4615,0.276696,0.235231,0.245189,0.234491,0.251257,0.275879,0.261799,0.312457,0.271462,0.25212,0.271521,0.327091,0.280333,0.299283,0.292117,0.295319,0.348106,0.334122,0.372043,0.467433,0.572772,0.702882,0.782884,0.83138,0.769291,0.694243,0.594376,0.518688,0.427573,0.423598,0.391337,0.458931,0.516676,0.631096,0.831176,1.007715,1.123969,1.273341,1.296887,1.285402,1.281348,1.267459,1.228319,1.168123,1.055914,0.968915,0.837402,0.732388,0.659242,0.603547,0.559538,0.522058,0.478491,0.472084,0.46243,0.487206,0.505809,0.497963,0.500566,0.497306
STD-11,0,InnoSpectra_NIR-S_demo,3,8.4615,0.269003,0.232247,0.240909,0.266818,0.25476,0.274673,0.257815,0.275329,0.274277,0.250819,0.28485,0.299591,0.269498,0.281904,0.299175,0.32101,0.313225,0.371761,0.371835,0.487377,0.544164,0.666043,0.734928,0.794901,0.75003,0.621846,0.504219,0.499386,0.403492,0.3592,0.379634,0.419826,0.518107,0.60562,0.755709,0.922731,1.108496,1.158195,1.238112,1.241263,1.215935,1.183455,1.156041,1.090931,1.02931,0.913785,0.763449,0.661019,0.566569,0.530994,0.468106,0.457803,0.427545,0.406361,0.393498,0.381925,0.419668,0.38895,0.425775,0.388115
STD-12,0,InnoSpectra_NIR-S_demo,1,9.2308,0.267097,0.244532,0.26995,0.3042,0.269668,0.253481,0.27191,0.262217,0.273507,0.310946,0.269175,0.268381,0.301516,0.28944,0.297877,0.296976,0.324888,0.372567,0.408864,0.473023,0.585144,0.694826,0.787046,0.84907,0.801879,0.707784,0.599845,0.490047,0.444693,0.433707,0.461991,0.475416,0.540012,0.638008,0.833068,1.002528,1.172405,1.262328,1.265064,1.280013,1.286632,1.233132,1.19373,1.112764,1.028626,0.950754,0.809703,0.724351,0.623175,0.546682,0.507102,0.479594,0.453316,0.445129,0.443188,0.430111,0.467586,0.451479,0.4449,0.440404
STD-12,0,InnoSpectra_NIR-S_demo,2,9.2308,0.269047,0.286571,0.316631,0.294947,0.304074,0.287667,0.315284,0.300156,0.302504,0.301798,0.315373,0.302633,0.316325,0.322941,0.313845,0.303409,0.356182,0.354026,0.391979,0.506494,0.616015,0.75406,0.848145,0.8199,0.806403,0.727173,0.590825,0.541408,0.471623,0.433911,0.445685,0.495532,0.592927,0.701772,0.808886,1.033789,1.159596,1.269093,1.276625,1.28468,1.274026,1.267538,1.202597,1.150485,1.036495,0.910929,0.839116,0.71877,0.624798,0.564017,0.526838,0.480912,0.453382,0.450077,0.438269,0.436953,0.473713,0.4614,0.472927,0.444524
STD-12,0,InnoSpectra_NIR-S_demo,3,9.2308,0.186429,0.172388,0.186908,0.170779,0.198358,0.18577,0.19621,0.210499,0.18993,0.18744,0.194758,0.221271,0.209068,0.235876,0.231099,0.247376,0.244705,0.288393,0.330758,0.367847,0.534996,0.644574,0.714656,0.732044,0.680464,0.621049,0.495403,0.399322,0.346191,0.315058,0.368065,0.373437,0.455283,0.57062,0.738147,0.920117,1.097776,1.161041,1.229386,1.163288,1.19396,1.199492,1.100475,1.064556,0.975118,0.82114,0.743295,0.606682,0.536911,0.462515,0.402655,0.369072,0.358567,0.332362,0.355939,0.353913,0.371224,0.361585,0.354423,0.354638
STD-13,0,InnoSpectra_NIR-S_demo,1,10.0,0.33449,0.344877,0.357057,0.319677,0.343652,0.365495,0.369019,0.380988,0.370719,0.364969,0.365198,0.390372,0.384026,0.397107,0.41839,0.423783,0.396614,0.467742,0.487637,0.566941,0.75285,0.852854,0.938588,0.9753,0.952509,0.867844,0.734901,0.606518,0.525936,0.517344,0.542114,0.566846,0.602745,0.741993,0.920688,1.093749,1.290816,1.386876,1.394404,1.356115,1.35664,1.342081,1.294412,1.241451,1.130105,1.017102,0.909939,0.793375,0.690459,0.646334,0.58919,0.573946,0.548726,0.533564,0.557209,0.544475,0.531687,0.589083,0.591868,0.580603
STD-13,0,InnoSpectra_NIR-S_demo,2,10.0,0.221604,0.206473,0.204464,0.22067,0.205813,0.209655,0.241234,0.232247,0.238988,0.238769,0.249059,0.248087,0.22332,0.236384,0.271162,0.252821,0.292367,0.286363,0.322368,0.436099,0.546616,0.672678,0.770585,0.780894,0.776982,0.660163,0.524126,0.408296,0.38004,0.341309,0.356032,0.382624,0.453935,0.55374,0.698365,0.92303,1.099827,1.145462,1.176759,1.198333,1.142382,1.165795,1.082802,0.995649,0.914262,0.799206,0.710706,0.554384,0.483961,0.413484,0.359703,0.345118,0.314499,0.314846,0.309687,0.336124,0.313908,0.333019,0.354456,0.355847
STD-13,0,InnoSpectra_NIR-S_demo,3,10.0,0.23571,0.240309,0.238345,0.25776,0.263432,0.239959,0.26338,0.265499,0.273901,0.254675,0.250918,0.271641,0.297522,0.266836,0.305407,0.297397,0.314822,0.344811,0.391527,0.483727,0.620178,0.765915,0.843791,0.881313,0.814517,0.722657,0.620958,0.519117,0.430993,0.408141,0.407034,0.433043,0.522237,0.621201,0.823438,0.960625,1.13281,1.227075,1.265368,1.252944,1.232977,1.17451,1.187469,1.083403,1.035508,0.902571,0.774975,0.679053,0.622327,0.520584,0.457454,0.400938,0.423061,0.405893,0.438619,0.427491,0.451358,0.435365,0.437464,0.476133
VINEGAR_WHITE_W4REF,0,InnoSpectra_NIR-S_demo,1,4.5,0.178137,0.154669,0.161778,0.184628,0.185654,0.191294,0.20202,0.216232,0.215744,0.219869,0.206541,0.275668,0.259813,0.276269,0.278126,0.304513,0.292937,0.327881,0.322275,0.367354,0.433698,0.492041,0.54653,0.531749,0.529685,0.468389,0.401835,0.400895,0.360965,0.344135,0.365788,0.42461,0.47711,0.568717,0.71131,0.85214,0.966715,1.119883,1.169222,1.21718,1.208178,1.190857,1.207614,1.125839,1.034728,0.92766,0.826095,0.731859,0.645254,0.542885,0.499817,0.484591,0.488489,0.474736,0.4442,0.433585,0.461309,0.457317,0.476437,0.483371
VINEGAR_WHITE_W4REF,0,InnoSpectra_NIR-S_demo,2,4.5,0.268229,0.28671,0.302288,0.296212,0.31246,0.2976,0.327065,0.31951,0.314096,0.313688,0.330604,0.341642,0.373863,0.355615,0.371085,0.330922,0.388858,0.383102,0.391318,0.405758,0.480484,0.576884,0.605425,0.617378,0.573952,0.545181,0.491258,0.44274,0.396855,0.398299,0.41867,0.509532,0.48437,0.636252,0.759178,0.889496,1.041279,1.140233,1.212444,1.236798,1.236265,1.309229,1.25237,1.123689,1.065008,0.924783,0.82506,0.695757,0.607995,0.589649,0.501387,0.474772,0.467742,0.469468,0.430889,0.447707,0.450549,0.429167,0.431554,0.454082
VINEGAR_WHITE_W4REF,0,InnoSpectra_NIR-S_demo,3,4.5,0.173996,0.218029,0.21168,0.195944,0.22305,0.20674,0.23157,0.262738,0.240295,0.271046,0.275004,0.271066,0.284017,0.301063,0.310895,0.304621,0.319549,0.349413,0.344047,0.380206,0.458467,0.518149,0.51541,0.570754,0.572471,0.518135,0.446323,0.417097,0.360623,0.397124,0.408983,0.455618,0.499468,0.606498,0.717243,0.895009,1.028409,1.099283,1.172685,1.255364,1.287751,1.289385,1.220483,1.159175,1.051212,0.93322,0.828103,0.751134,0.637659,0.609816,0.510804,0.496531,0.492849,0.483438,0.473619,0.443665,0.469792,0.501908,0.493426,0.495763
VINEGAR_B_DEMO,0,InnoSpectra_NIR-S_demo,1,6.2,0.312175,0.312154,0.309117,0.292569,0.320315,0.31271,0.317955,0.333383,0.354097,0.315487,0.333184,0.360214,0.313172,0.345894,0.334793,0.363666,0.391997,0.387571,0.444368,0.492783,0.598246,0.591647,0.698131,0.733777,0.710578,0.674261,0.591604,0.505177,0.450899,0.484801,0.482737,0.508757,0.589899,0.704847,0.821911,1.006249,1.101875,1.24776,1.326241,1.335217,1.318023,1.300486,1.28541,1.190979,1.097791,0.965563,0.866758,0.791732,0.684378,0.617565,0.562006,0.55074,0.540275,0.526013,0.471986,0.519492,0.504963,0.521894,0.508319,0.549742
VINEGAR_B_DEMO,0,InnoSpectra_NIR-S_demo,2,6.2,0.347761,0.326459,0.321991,0.296219,0.323944,0.331283,0.368967,0.329304,0.34901,0.359972,0.350728,0.34999,0.369542,0.340132,0.361061,0.373697,0.411518,0.431501,0.457243,0.522178,0.550865,0.675289,0.725643,0.731265,0.741912,0.661221,0.583261,0.504788,0.501101,0.509411,0.504701,0.549889,0.616686,0.699098,0.834729,0.9972,1.153117,1.250028,1.314135,1.33313,1.373909,1.332221,1.334188,1.225914,1.127524,1.035777,0.900972,0.780535,0.742183,0.687104,0.583407,0.594221,0.557441,0.553556,0.58018,0.530361,0.54658,0.57975,0.566283,0.573958
VINEGAR_B_DEMO,0,InnoSpectra_NIR-S_demo,3,6.2,0.225372,0.244897,0.236351,0.253657,0.293466,0.221262,0.257867,0.25325,0.237528,0.258449,0.239253,0.230316,0.253158,0.281427,0.255323,0.273706,0.268238,0.289161,0.31483,0.397791,0.483374,0.528093,0.587966,0.65004,0.601671,0.522244,0.485528,0.408476,0.367797,0.363225,0.364243,0.457352,0.433701,0.559617,0.718135,0.880175,1.002322,1.116514,1.172741,1.224431,1.241288,1.232131,1.165856,1.111429,0.998181,0.87214,0.758274,0.673076,0.581018,0.499683,0.444692,0.411967,0.432016,0.392884,0.393896,0.415253,0.416916,0.407326,0.402031,0.389385
"""

In [ ]:
df = load_spectra()
meta, X_all, wl = split_spectra(df)
print(f"讀入 {len(df)} 筆光譜，共 {meta['sample_id'].nunique()} 個樣品、"
      f"{len(wl)} 個波長點（{wl.min():.0f}-{wl.max():.0f} nm）。")
df.head()

### （選用）在 Colab 上傳班級實際資料

In [ ]:
# try:
#     from google.colab import files
#     uploaded = files.upload()
#     fname = list(uploaded.keys())[0]
#     df = load_spectra(fname)
#     meta, X_all, wl = split_spectra(df)
#     print(f"已改用上傳檔案：{fname}")
# except ImportError:
#     print("非 Colab 環境，略過上傳步驟；請改用 load_spectra('本機路徑或URL')")

## 2. 資料總覽 (Data Overview)

先畫幾條代表性的光譜，確認資料讀取正確：可以看到隨著醋酸濃度升高，1450 nm 附近的水 O–H 大帶形狀被逐漸疊加的醋酸訊號微幅改變。

In [ ]:
show_idx = [0, 1, 2, 3, 4, 5]  # a few standards spanning low -> high concentration
example_labels = [f"{meta.loc[i, 'sample_id']} ({meta.loc[i, 'acidity_pct']:.2f}%)" for i in show_idx]
ax = plot_spectra(wl, X_all[show_idx], labels=example_labels)
ax.set_title("Example raw NIR spectra (increasing acetic acid %)")
plt.tight_layout()
plt.show()

## 3. 校正集／測試集切分 (Calibration / Test Split)

**測試集只能用一次**，用來在最後誠實地評估模型；它絕對不能參與挑選前處理方法或潛在變數（LV）數，否則誤差估計會系統性偏樂觀（過度樂觀，optimistic bias）。

切分要在**樣品層級**（sample-level）進行，不是在光譜（重複）層級：同一個樣品的 3 次重複必須整個一起分到校正集或測試集，不能拆開——這正是後面第 4 節交叉驗證要用 `GroupKFold`/`LeaveOneGroupOut` 分組的同一個理由。

### Kennard–Stone 演算法

Kennard–Stone (KS) 是化學計量學裡常用的「代表性子集挑選」演算法，操作在 **X 空間**（這裡是光譜空間）：

1. 先挑出彼此歐氏距離（Euclidean distance）最遠的一對樣品（保證涵蓋光譜空間的兩個極端）。
2. 之後每一輪，從剩下的樣品中挑出「離目前已選集合的最小距離」最大的那一個（也就是離已選集合最遠的樣品），逐步把樣品空間填滿、涵蓋範圍。
3. 重複直到達到想要的校正集大小。

因為第一步保證挑出最極端的一對，KS 選出的校正集自然會涵蓋樣品範圍的兩端，不容易讓測試集需要「外插」（extrapolation）。

> **另一種常見做法：依濃度分層抽樣（stratified by concentration）**——當光譜只受單一分析物濃度支配（像本週的醋酸水溶液）時，KS 在 X 空間選出的順序通常會接近直接按濃度排序後分層抽樣的結果；但當光譜同時受多個成分或基質效應影響時，KS 在 X 空間工作能捕捉到單看濃度看不出來的差異，是更一般化的做法。本筆記本採用 KS，並在下面印出結果讓你對照兩者的差異。

**一個額外的設計選擇**：跟課堂 W13 deck 一樣，`VINEGAR_WHITE_W4REF`（W4 滴定的白醋，唯一一個「已知答案的真實樣品查核點」）**固定放進測試集**，不讓它參與 Kennard-Stone 的挑選——它的角色是「模型建好之後，拿一個有獨立參考值的真實樣品來驗收」，剩下的樣品才交給 KS 決定校正/測試的分配。

In [ ]:
def kennard_stone(X, n_select):
    """Kennard-Stone sample-selection algorithm.

    Greedily builds a subset of `n_select` rows of X that is spread out
    across the multivariate X-space: start from the most distant pair, then
    repeatedly add whichever remaining row has the largest *minimum*
    distance to the already-selected set. Returns the selected row indices
    in the order they were picked (index 0 and 1 are the initial extreme
    pair).
    """
    X = np.asarray(X, dtype=float)
    n = X.shape[0]
    D = squareform(pdist(X))
    i0, i1 = np.unravel_index(np.argmax(D), D.shape)
    selected = [int(i0), int(i1)]
    remaining = [i for i in range(n) if i not in selected]
    while len(selected) < n_select:
        min_d = D[np.ix_(remaining, selected)].min(axis=1)
        nxt = remaining[int(np.argmax(min_d))]
        selected.append(nxt)
        remaining.remove(nxt)
    return selected


# Run KS on the per-sample MEAN spectrum (one row per unique sample_id), so
# replicates are never split between calibration and test.
sample_table = (
    meta.assign(row=np.arange(len(meta)))
    .groupby("sample_id", sort=False)
    .agg(acidity_pct=("acidity_pct", "first"), rows=("row", list))
    .reset_index()
)
mean_spectra = np.array([X_all[r["rows"]].mean(axis=0) for _, r in sample_table.iterrows()])

N_UNIQUE = len(sample_table)

# The W4-titration reference vinegar is always held out as the independent,
# real-world "known answer" check sample (same design choice as the W13
# lab deck) -- it is never allowed into the pool Kennard-Stone selects from.
PINNED_TEST_ID = "VINEGAR_WHITE_W4REF"
pinned_pos = int(sample_table.index[sample_table["sample_id"] == PINNED_TEST_ID][0])
remaining_pos = [i for i in range(N_UNIQUE) if i != pinned_pos]

N_CAL_SAMPLES = int(round(0.72 * N_UNIQUE))  # ~72% of samples -> calibration
ks_local = kennard_stone(mean_spectra[remaining_pos], N_CAL_SAMPLES)
cal_sample_pos = [remaining_pos[j] for j in ks_local]
test_sample_pos = [pinned_pos] + [i for i in remaining_pos if i not in cal_sample_pos]

cal_sample_ids = set(sample_table.loc[cal_sample_pos, "sample_id"])
test_sample_ids = set(sample_table.loc[test_sample_pos, "sample_id"])

cal_mask = meta["sample_id"].isin(cal_sample_ids).to_numpy()
test_mask = meta["sample_id"].isin(test_sample_ids).to_numpy()

X_cal, y_cal, g_cal = X_all[cal_mask], meta.loc[cal_mask, "acidity_pct"].to_numpy(), meta.loc[cal_mask, "sample_id"].to_numpy()
X_test, y_test, g_test = X_all[test_mask], meta.loc[test_mask, "acidity_pct"].to_numpy(), meta.loc[test_mask, "sample_id"].to_numpy()

print(f"樣品總數 n={N_UNIQUE}；校正集樣品數={len(cal_sample_ids)}，測試集樣品數={len(test_sample_ids)}")
print(f"校正集光譜數={cal_mask.sum()}，測試集光譜數={test_mask.sum()}（測試集佔全部光譜 {test_mask.mean()*100:.1f}%）")
print("校正集樣品：", sorted(cal_sample_ids))
print("測試集樣品：", sorted(test_sample_ids))

assert 0.20 <= test_mask.mean() <= 0.35, "測試集比例應落在 20-35% 之間"
assert sample_table["acidity_pct"].min() in meta.loc[cal_mask, "acidity_pct"].values, "校正集應涵蓋濃度下限，避免測試集外插"
assert sample_table["acidity_pct"].max() in meta.loc[cal_mask, "acidity_pct"].values, "校正集應涵蓋濃度上限，避免測試集外插"
print("檢查通過：校正集涵蓋了濃度的兩個極端，測試集比例落在合理範圍。")

## 4. 前處理比較 (Preprocessing Comparison)

比較三種前處理方法對 **RMSECV**（交叉驗證均方根誤差）的影響：

- **raw**：不做額外前處理（PLS 內部仍會用 sklearn 預設的「平均置中＋autoscale」）。
- **SNV**（Standard Normal Variate）：每條光譜各自減去自己的平均值、除以自己的標準差，扣掉整體散射造成的乘性/加性效應。
- **SG 一階導數**（Savitzky–Golay 1st derivative）：用局部多項式擬合再微分，扣掉平緩的基線飄移，只保留形狀變化較快的訊號。

**關鍵規則：交叉驗證一定要用 `GroupKFold` 或 `LeaveOneGroupOut`，以 `sample_id` 分組**——如果直接對所有光譜（含重複）做普通的 `KFold`，同一樣品的其中 2 次重複可能進訓練折、第 3 次重複進驗證折，驗證折等於已經看過幾乎一樣的光譜，RMSECV 會系統性被低估（過度樂觀）。第 8 節的練習會實際示範這個資料外洩（data leakage）現象。

> 課堂 W13 deck（OR13）現在直接用的就是**本筆記本這批資料與這個流程**（SNV → PLS，LV=2），兩者的 RMSECV／RMSEP／RPD 應該完全一致，可以互相對答案。這裡仍然保留完整的三種前處理比較，是因為「前處理該選哪一種要用 RMSECV 實際比較，不能直接照抄別人的結論」——換一批資料（例如你們班自己量的光譜）重跑一次，選出來的最佳前處理完全可能不是 SNV。

In [ ]:
def snv(X):
    m = X.mean(axis=1, keepdims=True)
    s = X.std(axis=1, keepdims=True)
    return (X - m) / s


def sg_deriv1(X, window=9, polyorder=2):
    return savgol_filter(X, window_length=window, polyorder=polyorder, deriv=1, axis=1)


PREP = {
    "raw": (X_cal, X_test),
    "SNV": (snv(X_cal), snv(X_test)),
    "SG_1st_deriv": (sg_deriv1(X_cal), sg_deriv1(X_test)),
}

MAX_LV = 15


def rmsecv_curve_grouped(Xc, yc, groups, max_lv):
    """RMSEC and RMSECV (LeaveOneGroupOut by `groups`) for LV = 1..max_lv."""
    logo = LeaveOneGroupOut()
    out = []
    for nlv in range(1, max_lv + 1):
        pred = np.zeros_like(yc, dtype=float)
        for tr, va in logo.split(Xc, yc, groups):
            m = PLSRegression(n_components=nlv)
            m.fit(Xc[tr], yc[tr])
            pred[va] = m.predict(Xc[va]).ravel()
        rmsecv = float(np.sqrt(np.mean((pred - yc) ** 2)))
        m_full = PLSRegression(n_components=nlv).fit(Xc, yc)
        rmsec = float(np.sqrt(np.mean((m_full.predict(Xc).ravel() - yc) ** 2)))
        out.append((nlv, rmsec, rmsecv))
    return out


results_by_prep = {name: rmsecv_curve_grouped(Xc, y_cal, g_cal, MAX_LV) for name, (Xc, Xt) in PREP.items()}

print("== 各前處理法 RMSECV 最小值 ==")
best_prep, best_lv, best_rmsecv = None, None, np.inf
for name, curve in results_by_prep.items():
    lv_min = min(curve, key=lambda r: r[2])
    print(f"  {name:<14s}: 最小 RMSECV={lv_min[2]:.4f}  (LV={lv_min[0]})")
    if lv_min[2] < best_rmsecv:
        best_prep, best_lv, best_rmsecv = name, lv_min[0], lv_min[2]
print(f"選用前處理：{best_prep}，最佳 LV={best_lv}，RMSECV={best_rmsecv:.4f}")

## 5. RMSEC / RMSECV vs LV 數

用選定的前處理法，畫出 RMSEC（校正集配適誤差，只會隨 LV 增加持續下降）與 RMSECV（分組交叉驗證誤差，會在某個 LV 附近出現最小值）的曲線。RMSEC 與 RMSECV 分開拉大的地方，就是模型開始過擬合（overfitting）的訊號。

In [ ]:
Xc_best, Xt_best = PREP[best_prep]
curve = results_by_prep[best_prep]
lv_list = [r[0] for r in curve]
rmsec_list = [r[1] for r in curve]
rmsecv_list = [r[2] for r in curve]

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(lv_list, rmsec_list, "o-", color="tab:blue", label="RMSEC (calibration fit)")
ax.plot(lv_list, rmsecv_list, "s-", color="tab:orange", label="RMSECV (LeaveOneGroupOut CV)")
ax.axvline(best_lv, color="grey", linestyle="--", linewidth=1, label=f"chosen LV = {best_lv}")
ax.set_xlabel("Number of latent variables (LV)")
ax.set_ylabel("RMSE (% w/v)")
ax.set_title(f"RMSEC vs RMSECV  (preprocessing = {best_prep})")
ax.legend()
plt.tight_layout()
plt.show()

## 6. 最終模型與品質指標 (Final Model & Figures of Merit)

用選定的前處理法與 LV 數，在**整個校正集**上重新配適一次最終模型，然後**只用一次**測試集算出：

- **RMSEP**（測試集均方根誤差）
- **R²**（模型解釋參考值變異的比例）
- **bias**（測試集預測值的平均系統性偏差）
- **RPD** = SD(y_test) / RMSEP（相對於測試集本身變異的表現；全課程統一分級：< 2 不足、2–2.5 粗篩、2.5–3 篩選、3–5 品管定量、> 5 優良）

因為 y（`acidity_pct`）本身帶有約 `PREP_SD` = 0.05% w/v 的滴定參考值不確定度，**RMSEP 理論上很難顯著小於這個數字**——這是參考方法精密度設下的天花板。

In [ ]:
final_model = PLSRegression(n_components=best_lv).fit(Xc_best, y_cal)
pred_test = final_model.predict(Xt_best).ravel()

RMSEP = float(np.sqrt(np.mean((pred_test - y_test) ** 2)))
resid_test = pred_test - y_test
bias_test = float(np.mean(resid_test))
ss_res = float(np.sum((y_test - pred_test) ** 2))
ss_tot = float(np.sum((y_test - y_test.mean()) ** 2))
R2_test = 1 - ss_res / ss_tot
SD_ytest = float(np.std(y_test, ddof=1))
RPD = SD_ytest / RMSEP

RMSEC_final = float(np.sqrt(np.mean((final_model.predict(Xc_best).ravel() - y_cal) ** 2)))
RMSECV_final = best_rmsecv

print(f"前處理={best_prep}  LV={best_lv}")
print(f"RMSEC={RMSEC_final:.4f}  RMSECV={RMSECV_final:.4f}  RMSEP={RMSEP:.4f}  "
      f"R2(test)={R2_test:.4f}  bias(test)={bias_test:+.4f}  RPD={RPD:.2f}")
print(f"參考值不確定度 PREP_SD={PREP_SD:.2f}%  ->  RMSEP 理論下限大約在這個量級附近")

In [ ]:
axis_lo = min(y_cal.min(), y_test.min(), pred_test.min()) - 0.5
axis_hi = max(y_cal.max(), y_test.max(), pred_test.max()) + 0.5

# Calibration-side points are shown using LeaveOneGroupOut CV predictions at
# the chosen LV, to avoid the optimistic bias of "fit and predict on the same
# calibration data".
logo = LeaveOneGroupOut()
cv_pred_cal = np.zeros_like(y_cal, dtype=float)
for tr, va in logo.split(Xc_best, y_cal, g_cal):
    m = PLSRegression(n_components=best_lv).fit(Xc_best[tr], y_cal[tr])
    cv_pred_cal[va] = m.predict(Xc_best[va]).ravel()

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.scatter(y_cal, cv_pred_cal, s=45, color="tab:blue", alpha=0.8, label="Calibration (CV predicted)")
ax.scatter(y_test, pred_test, s=70, color="tab:orange", marker="^", label="Test (independent)")
ax.plot([axis_lo, axis_hi], [axis_lo, axis_hi], "--", color="grey", linewidth=1.5, label="1:1 line")
ax.set_xlim(axis_lo, axis_hi)
ax.set_ylim(axis_lo, axis_hi)
ax.set_xlabel("Reference acidity (%w/v, W4 titration)")
ax.set_ylabel("PLS-predicted acidity (%w/v)")
ax.set_title("Predicted vs reference")
ax.legend()
ax.set_aspect("equal", adjustable="box")
plt.tight_layout()
plt.show()

## 7. Model Report Card

把這個模型的所有關鍵資訊整理成一張表，方便之後（W14／W17）快速對照與比較。

In [ ]:
report_card = pd.DataFrame([{
    "instrument": meta.loc[cal_mask, "instrument"].iloc[0],
    "preprocessing": best_prep,
    "n_LV": best_lv,
    "n_calibration_spectra": int(cal_mask.sum()),
    "n_test_spectra": int(test_mask.sum()),
    "RMSEC_pct": round(RMSEC_final, 4),
    "RMSECV_pct": round(RMSECV_final, 4),
    "RMSEP_pct": round(RMSEP, 4),
    "R2_test": round(R2_test, 4),
    "bias_pct": round(bias_test, 4),
    "RPD": round(RPD, 2),
    "applicable_range_pct": f"{y_cal.min():.1f}-{y_cal.max():.1f}",
}])
report_card

## 8. 其他工具的等效寫法 (R / Orange)

### R（`pls` 套件）

```r
library(pls)
# segments= 要給的是「每一折包含哪些列索引」的 list，不是分組向量本身；
# split() 把列索引依 sample_id 分組，正是 GroupKFold/LeaveOneGroupOut 的等效寫法。
seg_by_sample <- split(seq_len(nrow(cal_df)), cal_df$sample_id)
mod <- plsr(y ~ X, ncomp = 15, data = cal_df, validation = "CV",
            segments = seg_by_sample)
plot(RMSEP(mod))                            # 找 RMSECV 最小值對應的 ncomp
pred <- predict(mod, ncomp = best_lv, newdata = test_df)
```

`validation="CV"` 搭配 `segments=`（一個 list，每個元素是同一組要放在同一折的列索引）效果等同本筆記本用 `LeaveOneGroupOut`／`GroupKFold` 以 `sample_id` 分組交叉驗證，避免同一樣品的重複同時出現在訓練與驗證。

### Orange（OR13：兩個 `Test and Score`）

OR13 的流程是 `File`（讀入光譜＋參考值）→ `SNV` → `Select Rows`（切成校正集／測試集）→ `PLS` → **兩個** `Test and Score`：

- **Test and Score ①**：只接校正集，交叉驗證方式選 **Cross validation by feature → `sample_id`**（以樣品分組，等同本筆記本的 `LeaveOneGroupOut`/`GroupKFold`）。這個表格的 RMSE 欄就是 **RMSECV**，用來挑 LV。
- **Test and Score ②**：另外接測試集，評估方式選 **Test on test data**。這個表格的 RMSE 欄就是**獨立**的 **RMSEP**——不是用 `Predictions` 元件，因為 `Test and Score` 本身在「Test on test data」模式下就會直接算出測試集的誤差指標（RMSE、R²…）。

邏輯與本筆記本完全一致：**配適 → 分組交叉驗證選 LV → 用固定模型預測測試集**，只是介面不同、且 Orange 把這兩步分別放進兩個獨立的 `Test and Score` 元件裡。

## 9. 載入真實 InnoSpectra／Avantes 匯出資料

只要把班上實際量測的資料整理成本筆記本開頭定義的共用長格式（`sample_id, group, instrument, rep, acidity_pct, <波長...>`），存成 CSV，就能直接用 `load_spectra()` 載入、套用第 3–7 節完全一樣的流程。

若你的資料是用 **SpectraView** 匯出（`layout="rows"`），`load_spectra()` 讀進來後 `split_spectra()` 只會給你 `sample_id`——沒有 `acidity_pct` 這類標籤，因為 SpectraView 的匯出格式本來就不帶這些資訊。這時候需要自己準備一張「`sample_id` → `acidity_pct`」的對照表（例如另存一份小 CSV，或直接在筆記本裡手打一個 dict），再用 `sample_id` 做 `merge`，如下面範例所示。

In [ ]:
# 範例 A：資料已經是本課程的共用長格式（含 acidity_pct 等標籤），直接載入即可：
#
# real_df = load_spectra("data/w13_class_nir_export.csv")
# real_meta, real_X, real_wl = split_spectra(real_df)
# plot_spectra(real_wl, real_X[:5], labels=real_meta["sample_id"].iloc[:5].tolist())
#
# 或直接餵一個已發布的 Google 試算表 CSV 連結：
# real_df = load_spectra("https://.../pub?output=csv")

# 範例 B：資料是 SpectraView 的 "rows" 匯出檔（只有 sample_id，沒有 acidity_pct），
# 需要自己補一張標籤對照表，用 sample_id 合併：
#
# sv_df = load_spectra("data/spectraview_export.csv")
# sv_meta, sv_X, sv_wl = split_spectra(sv_df)   # sv_meta 只有一欄：sample_id
# label_lookup = pd.DataFrame({
#     "sample_id": ["STD-00", "STD-01", "..."],
#     "acidity_pct": [0.0, 0.77, ...],
# })
# sv_meta = sv_meta.merge(label_lookup, on="sample_id", how="left")

print("此 cell 僅示範用法（見上方註解），因為示範環境沒有實際的班級匯出檔可供載入。")
print("實際使用時，把 load_spectra() 的路徑換成你們班的匯出 CSV，並視資料來源選擇範例 A 或 B。")

## 10. 練習題 (Exercises)

請在每題下方的空白程式碼區塊中作答（可以用文字 print 說明，也可以搭配計算/繪圖）。

### 問題 1

比較三種前處理法在最佳 LV 時的 RMSECV（見 `results_by_prep`），哪一種對這批模擬資料最好？試著解釋為什麼（提示：這批資料的「雜訊」主要是哪一種型態：整體散射／基線飄移，還是隨機雜訊？）。

In [ ]:
# TODO: 請在這裡作答

### 問題 2（資料外洩示範）

把第 4 節的分組交叉驗證（`LeaveOneGroupOut`，以 `sample_id` 分組）換成**不分組**的普通 `KFold`（例如 5 折），對校正集重新算一次 RMSECV（固定用 `best_prep` 與 `best_lv`）。比較兩者的數字，並解釋為什麼不分組會讓 RMSECV 看起來比較樂觀。

In [ ]:
# TODO: 請在這裡作答

### 問題 3

把第 3 節的 Kennard–Stone 切分，改成單純依 `acidity_pct` 由小到大排序後的分層抽樣（例如每 4 個樣品取 1 個當測試集）。比較新切分下的測試集組成（涵蓋的濃度範圍）與最終 RMSEP，跟原本 KS 切分的結果有什麼不同？

In [ ]:
# TODO: 請在這裡作答

### 問題 4

本模型的 RPD 大約是多少？對照課堂統一的 RPD 分級（< 2 不足、2–2.5 粗篩、2.5–3 篩選、3–5 品管定量、> 5 優良），這個模型落在哪一級、適合拿來做哪一種等級的應用？如果落在「品管定量」以下的等級，可能可以怎麼改善（前處理？增加校正集樣品數？換更寬的濃度範圍？）？

In [ ]:
# TODO: 請在這裡作答

### 問題 5

`PREP_SD`（W4 滴定參考值不確定度）設定為 0.05% w/v。如果把它改成 0.20%（重新產生資料、重跑整個流程），你預期 RMSEP 會怎麼變化？為什麼參考方法本身的精密度會限制模型 RMSEP 的下限？

In [ ]:
# TODO: 請在這裡作答

## 11. 匯出結果 (Export)

把 model report card 存成 CSV。若在 Colab 環境執行，會另外觸發瀏覽器下載；在本機執行則只會存檔。

In [ ]:
output_path = "nb11_model_report_card.csv"
report_card.to_csv(output_path, index=False)
print(f"已儲存：{output_path}")

try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print("非 Colab 環境，略過自動下載（檔案已存在本機/雲端硬碟工作目錄）。")